# Hybrid Bio-Extraction: GLiNER2 + REBEL (V2 - Robust)
## Span-Anchored Relation Extraction

This notebook implements a hybrid pipeline that combines **GLiNER2** (Entity Grounding) and **REBEL** (Relation Discovery).

**Update**: Added truncation and error handling to handle biographies longer than 1024 tokens and prevent CUDA crashes.

In [ ]:
!pip install -q gliner2 transformers torch tqdm

In [ ]:
import torch
import json
from gliner2 import GLiNER2
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from tqdm.auto import tqdm
from pathlib import Path

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

# --- CONFIGURATION ---
INPUT_DATA_PATH = "/kaggle/input/datasets/pallamreddyviswas/train-set-wikibio/openie_input.txt"
OUTPUT_FILE = "/kaggle/working/hybrid_extractions.json"
PROCESS_LIMIT = 2000 # Set to None to process everything

## 1. Initialize Models

In [ ]:
print("Loading GLiNER2...")
gliner_model = GLiNER2.from_pretrained("fastino/gliner2-base-v1").to(DEVICE)
gliner_labels = ["person", "organization", "occupation", "location", "education", "award"]

print("Loading REBEL...")
rebel_tokenizer = AutoTokenizer.from_pretrained("Babelscape/rebel-large")
rebel_model = AutoModelForSeq2SeqLM.from_pretrained("Babelscape/rebel-large").to(DEVICE)

## 2. Extraction & Alignment Logic

In [ ]:
def extract_rebel_triplets(text, model, tokenizer):
    """Extract and parse triplets from REBEL output tokens with truncation fix."""
    try:
        # FIX: Added truncation=True and max_length=1024 to prevent CUDA error on very long bios
        inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=1024).to(DEVICE)
        
        gen_kwargs = {"max_length": 256, "length_penalty": 0, "num_beams": 3, "num_return_sequences": 1}
        generated_tokens = model.generate(inputs["input_ids"], **gen_kwargs)
        decoded = tokenizer.batch_decode(generated_tokens, skip_special_tokens=False)[0]
        
        triplets = []
        text = decoded.replace("<s>", "").replace("</s>", "").replace("<pad>", "")
        parts = text.split("<triplet>")
        for part in parts:
            if not part.strip(): continue
            rel_split = part.split("<rel>")
            if len(rel_split) < 2: continue
            subject = rel_split[0].strip()
            for rel_part in rel_split[1:]:
                obj_split = rel_part.split("<obj>")
                if len(obj_split) < 2: continue
                relation = obj_split[0].strip()
                for obj in obj_split[1:]:
                    triplets.append({"subj": subject, "rel": relation, "obj": obj.strip()})
        return triplets
    except Exception as e:
        print(f"Skipping REBEL extraction for this entry due to error: {e}")
        return []

def align_triplets(rebel_triples, gliner_entities):
    cleaned = []
    for trip in rebel_triples:
        s, r, o = trip['subj'], trip['rel'], trip['obj']
        for ent in gliner_entities:
            if s.lower() in ent['text'].lower() or ent['text'].lower() in s.lower():
                s = ent['text']
            if o.lower() in ent['text'].lower() or ent['text'].lower() in o.lower():
                o = ent['text']
        cleaned.append({"subject": s, "relation": r, "object": o})
    return cleaned

## 3. Full Dataset Processing

In [ ]:
def run_full_pipeline(input_path, output_path, limit=None):
    print(f"Reading data from {input_path}...")
    if not Path(input_path).exists():
        print(f"ERROR: {input_path} not found.")
        return

    with open(input_path, 'r', encoding='utf-8') as f:
        bios = [line.strip() for line in f if line.strip()]
    
    if limit:
        bios = bios[:limit]
        
    results = []
    for bio in tqdm(bios, desc="Processing Biographies"):
        try:
            # 1. GLiNER Pass
            gliner_res = gliner_model.extract_entities(bio, gliner_labels)
            
            # 2. REBEL Pass
            re_triples = extract_rebel_triplets(bio, rebel_model, rebel_tokenizer)
            
            # 3. Alignment
            final_triples = align_triplets(re_triples, gliner_res)
            
            results.append({
                "text": bio,
                "entities": gliner_res,
                "triples": final_triples
            })
        except Exception as e:
            print(f"Fatal error processing biography: {e}")
            continue
        
    # Save Results
    with open(output_path, 'w', encoding='utf-8') as f:
        json.dump(results, f, indent=2)
    print(f"\nProcessing complete! Results saved to {output_path}")

# --- START RUN ---
run_full_pipeline(INPUT_DATA_PATH, OUTPUT_FILE, limit=PROCESS_LIMIT)